<a href="https://colab.research.google.com/github/poetswentodie/AI-Chatbot-Python/blob/main/coa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Python Branch Predictor Simulation

This section translates the provided JavaScript branch predictor simulation logic into Python. It includes a seedable random number generator, various branch predictor implementations, trace generation functions, and a simulation pipeline to evaluate predictor performance.

In [2]:
import math
import array

# ---------------------------------------------------------------
# Seeded RNG (mulberry32) so traces are reproducible
# ---------------------------------------------------------------
class Mulberry32:
    def __init__(self, seed):
        self.seed = seed

    def random(self):
        self.seed |= 0
        self.seed = (self.seed + 0x6D2B79F5) & 0xFFFFFFFF
        t = math.imul(self.seed ^ (self.seed >> 15), 1 | self.seed)
        t = (t + math.imul(t ^ (t >> 7), 61 | t)) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296.0

### Branch Predictor Classes

This section defines the base `Predictor` class and various implementations, from simple static predictors to more complex history-based and AI-driven predictors.

In [3]:
class Predictor:
    def __init__(self, name):
        self.name = name
        self.correct = 0
        self.total = 0
        self.category = "base" # Default, will be overridden

    def predict(self, pc, offset):
        raise NotImplementedError("predict method not implemented")

    def update(self, pc, actual, predicted):
        raise NotImplementedError("update method not implemented")

    def evaluate(self, pc, actual, offset):
        predicted = self.predict(pc, offset if offset is not None else 0)
        self.total += 1
        if predicted == actual:
            self.correct += 1
        self.update(pc, actual, predicted)
        return predicted

    def accuracy(self):
        return self.correct / self.total if self.total else 0

    def reset(self):
        self.correct = 0
        self.total = 0

class AlwaysTaken(Predictor):
    def __init__(self):
        super().__init__("Always Taken")
        self.category = "static"
    def predict(self, pc, offset): return True
    def update(self, pc, actual, predicted): pass

class AlwaysNotTaken(Predictor):
    def __init__(self):
        super().__init__("Always Not Taken")
        self.category = "static"
    def predict(self, pc, offset): return False
    def update(self, pc, actual, predicted): pass

class BTFNT(Predictor):
    def __init__(self):
        super().__init__("BTFNT (Static)")
        self.category = "static"
    def predict(self, pc, offset): return offset < 0
    def update(self, pc, actual, predicted): pass

class OneBit(Predictor):
    def __init__(self, bits=10):
        super().__init__("1-Bit Predictor")
        self.category = "counter"
        self.size = 1 << bits
        self.table = array.array('B', [0] * self.size) # Uint8Array

    def idx(self, pc): return pc % self.size
    def predict(self, pc): return self.table[self.idx(pc)] == 1
    def update(self, pc, actual, predicted): self.table[self.idx(pc)] = 1 if actual else 0

class TwoBit(Predictor):
    def __init__(self, bits=10):
        super().__init__("2-Bit Saturating Counter")
        self.category = "counter"
        self.size = 1 << bits
        self.table = array.array('B', [1] * self.size) # Uint8Array, filled with 1

    def idx(self, pc): return pc % self.size
    def predict(self, pc): return self.table[self.idx(pc)] >= 2
    def update(self, pc, actual, predicted):
        i = self.idx(pc)
        self.table[i] = min(3, self.table[i] + 1) if actual else max(0, self.table[i] - 1)

class TwoLevelAdaptive(Predictor):
    def __init__(self, pc_bits=10, hist_bits=8):
        super().__init__("Two-Level Adaptive")
        self.category = "history"
        self.pc_size = 1 << pc_bits
        self.hist_bits = hist_bits
        self.hist_mask = (1 << hist_bits) - 1
        self.local_hist = array.array('H', [0] * self.pc_size) # Uint16Array
        self.pht = array.array('B', [1] * (1 << hist_bits)) # Uint8Array, filled with 1

    def pidx(self, pc): return pc % self.pc_size
    def predict(self, pc): return self.pht[self.local_hist[self.pidx(pc)]] >= 2
    def update(self, pc, actual, predicted):
        p = self.pidx(pc)
        h = self.local_hist[p]
        self.pht[h] = min(3, self.pht[h] + 1) if actual else max(0, self.pht[h] - 1)
        self.local_hist[p] = ((h << 1) | (1 if actual else 0)) & self.hist_mask

class GShare(Predictor):
    def __init__(self, table_bits=12, hist_bits=12):
        super().__init__("GShare")
        self.category = "history"
        self.ghr = 0
        self.ghr_mask = (1 << hist_bits) - 1
        self.pht = array.array('B', [1] * (1 << table_bits)) # Uint8Array, filled with 1
        self.idx_mask = (1 << table_bits) - 1

    def idx(self, pc): return (pc ^ self.ghr) & self.idx_mask
    def predict(self, pc): return self.pht[self.idx(pc)] >= 2
    def update(self, pc, actual, predicted):
        i = self.idx(pc)
        self.pht[i] = min(3, self.pht[i] + 1) if actual else max(0, self.pht[i] - 1)
        self.ghr = ((self.ghr << 1) | (1 if actual else 0)) & self.ghr_mask

class Tournament(Predictor):
    def __init__(self, table_bits=12, hist_bits=12):
        super().__init__("Tournament (Hybrid)")
        self.category = "history"
        self.local = TwoLevelAdaptive(table_bits, 10)
        self.gshare = GShare(table_bits, hist_bits)
        self.meta = array.array('B', [1] * (1 << hist_bits)) # Uint8Array, filled with 1
        self.meta_mask = (1 << hist_bits) - 1
        self._l_local = False # Internal state for update
        self._l_gshare = False # Internal state for update

    def midx(self): return self.gshare.ghr & self.meta_mask
    def predict(self, pc):
        self._l_local = self.local.predict(pc)
        self._l_gshare = self.gshare.predict(pc)
        return self.meta[self.midx()] >= 2 if self._l_gshare else self._l_local

    def update(self, pc, actual, predicted):
        m = self.midx()
        lc = self._l_local == actual
        gc = self._l_gshare == actual

        if gc and not lc:
            self.meta[m] = min(3, self.meta[m] + 1)
        elif lc and not gc:
            self.meta[m] = max(0, self.meta[m] - 1)

        self.local.update(pc, actual, self._l_local)
        self.gshare.update(pc, actual, self._l_gshare)

class PerceptronPredictor(Predictor):
    def __init__(self, table_size=512, hist_len=24):
        super().__init__("Perceptron (AI)")
        self.category = "ai"
        self.table_size = table_size
        self.hist_len = hist_len
        # List of array.arrays for weights
        self.weights = [array.array('h', [0] * (hist_len + 1)) for _ in range(table_size)] # Int16Array
        self.ghr = array.array('b', [0] * hist_len) # Int8Array
        self.theta = math.floor(1.93 * hist_len + 14)
        self._y = 0 # Internal state for update

    def idx(self, pc): return pc % self.table_size

    def dot(self, w):
        y = w[0]
        for i in range(self.hist_len):
            y += w[i + 1] * self.ghr[i]
        return y

    def predict(self, pc):
        w = self.weights[self.idx(pc)]
        self._y = self.dot(w)
        return self._y >= 0

    def update(self, pc, actual, predicted):
        w = self.weights[self.idx(pc)]
        t = 1 if actual else -1

        if predicted != actual or abs(self._y) <= self.theta:
            w[0] += t
            for i in range(self.hist_len):
                w[i + 1] += t * self.ghr[i]

        for i in range(self.hist_len - 1, 0, -1):
            self.ghr[i] = self.ghr[i - 1]
        self.ghr[0] = t

class NeuralMLP(Predictor):
    def __init__(self, hist_len=16, hidden=8, lr=0.15, seed=42):
        super().__init__("Neural MLP (AI)")
        self.category = "ai"
        self.h = hist_len
        self.hidden = hidden
        self.lr = lr
        self.rng = Mulberry32(seed)

        self.W1 = [[(self.rng.random() - 0.5) * 0.6 for _ in range(hist_len)] for _ in range(hidden)]
        self.b1 = array.array('d', [0.0] * hidden) # Float64Array
        self.W2 = array.array('d', [(self.rng.random() - 0.5) * 0.6 for _ in range(hidden)]) # Float64Array
        self.b2 = 0.0
        self.ghr = array.array('d', [0.0] * hist_len) # Float64Array
        self._a1 = None # Internal state for update
        self._y = None # Internal state for update

    def sigmoid(self, x):
        if x < -60: return 0.0
        if x > 60: return 1.0
        return 1 / (1 + math.exp(-x))

    def forward(self):
        a1 = array.array('d', [0.0] * self.hidden) # Float64Array
        for j in range(self.hidden):
            s = self.b1[j]
            row = self.W1[j]
            for i in range(self.h):
                s += row[i] * self.ghr[i]
            a1[j] = math.tanh(s)

        z2 = self.b2
        for j in range(self.hidden):
            z2 += self.W2[j] * a1[j]
        return {'a1': a1, 'y': self.sigmoid(z2)}

    def predict(self, pc, offset=None):
        result = self.forward()
        self._a1 = result['a1']
        self._y = result['y']
        return self._y >= 0.5

    def update(self, pc, actual, predicted):
        target = 1.0 if actual else 0.0
        err = self._y - target

        for j in range(self.hidden):
            self.W2[j] -= self.lr * err * self._a1[j]
        self.b2 -= self.lr * err

        for j in range(self.hidden):
            d = err * self.W2[j] * (1 - self._a1[j] * self._a1[j])
            for i in range(self.h):
                self.W1[j][i] -= self.lr * d * self.ghr[i]
            self.b1[j] -= self.lr * d

        t = 1.0 if actual else -1.0
        for i in range(self.h - 1, 0, -1):
            self.ghr[i] = self.ghr[i - 1]
        self.ghr[0] = t

def build_predictor_suite():
    return [
        AlwaysTaken(), AlwaysNotTaken(), BTFNT(),
        OneBit(), TwoBit(),
        TwoLevelAdaptive(), GShare(), Tournament(),
        PerceptronPredictor(), NeuralMLP(),
    ]

### Trace Generators

These functions create different branch traces to simulate various program behaviors, such as loops, alternating branches, and random patterns.

In [4]:
def tight_loop(iterations, body_len=4, base_pc=0x1000):
    trace = []
    pc = base_pc
    branch_pc = base_pc + body_len * 4
    for i in range(iterations):
        for k in range(body_len):
            trace.append({'pc': pc, 'isBranch': False})
            pc += 4
        trace.append({'pc': branch_pc, 'isBranch': True, 'taken': i < iterations - 1, 'offset': -body_len * 4})
        pc = branch_pc + 4
    return trace

def nested_loop(outer, inner, body_len=3, base_pc=0x2000):
    trace = []
    pc = base_pc
    outer_pc = base_pc + 0x100
    inner_pc = base_pc + 0x040
    for i in range(outer):
        for j in range(inner):
            for k in range(body_len):
                trace.append({'pc': pc, 'isBranch': False})
                pc += 4
            trace.append({'pc': inner_pc, 'isBranch': True, 'taken': j < inner - 1, 'offset': -body_len * 4})
        trace.append({'pc': outer_pc, 'isBranch': True, 'taken': i < outer - 1, 'offset': -(inner * (body_len + 1)) * 4})
    return trace

def alternating_branch(length, base_pc=0x3000, body_len=2):
    trace = []
    pc = base_pc
    branch_pc = base_pc + body_len * 4
    for i in range(length):
        for k in range(body_len):
            trace.append({'pc': pc, 'isBranch': False})
            pc += 4
        trace.append({'pc': branch_pc, 'isBranch': True, 'taken': i % 2 == 0, 'offset': -body_len * 4})
        pc = branch_pc + 4
    return trace

def periodic_pattern(length, pattern, base_pc=0x4000, body_len=2):
    trace = []
    pc = base_pc
    branch_pc = base_pc + body_len * 4
    for i in range(length):
        for k in range(body_len):
            trace.append({'pc': pc, 'isBranch': False})
            pc += 4
        trace.append({'pc': branch_pc, 'isBranch': True, 'taken': pattern[i % len(pattern)], 'offset': -body_len * 4})
        pc = branch_pc + 4
    return trace

def random_branch(length, p_taken=0.5, base_pc=0x5000, body_len=2, seed=1):
    rng = Mulberry32(seed)
    trace = []
    pc = base_pc
    branch_pc = base_pc + body_len * 4
    for i in range(length):
        for k in range(body_len):
            trace.append({'pc': pc, 'isBranch': False})
            pc += 4
        trace.append({'pc': branch_pc, 'isBranch': True, 'taken': rng.random() < p_taken, 'offset': body_len * 4})
        pc = branch_pc + 4
    return trace

def matrix_multiply_pattern(n, base_pc=0x6000):
    trace = []
    pc = base_pc
    pc_i = base_pc + 0x300
    pc_j = base_pc + 0x200
    pc_k = base_pc + 0x100
    for i in range(n):
        for j in range(n):
            for k in range(n):
                trace.append({'pc': pc, 'isBranch': False}); pc += 4
                trace.append({'pc': pc, 'isBranch': False}); pc += 4
                trace.append({'pc': pc_k, 'isBranch': True, 'taken': k < n - 1, 'offset': -8})
            trace.append({'pc': pc_j, 'isBranch': True, 'taken': j < n - 1, 'offset': -8 * n})
        trace.append({'pc': pc_i, 'isBranch': True, 'taken': i < n - 1, 'offset': -8 * n * n})
    return trace

def mixed_workload(total_length=4000, seed=7):
    chunk = math.floor(total_length / 6)
    trace = []
    trace.extend(tight_loop(math.floor(chunk / 4), 4, 0x1000))
    s = math.floor(math.sqrt(chunk / 6)) + 2
    trace.extend(nested_loop(s, s, 3, 0x2000))
    trace.extend(alternating_branch(chunk, 0x3000))
    trace.extend(periodic_pattern(chunk, [True, True, False], 0x4000))
    trace.extend(random_branch(chunk, 0.5, 0x5000, 2, seed))
    trace.extend(random_branch(chunk, 0.85, 0x5500, 2, seed + 1))
    return trace

def standard_benchmark_suite():
    return {
        "Tight Loop": tight_loop(1500, 4),
        "Nested Loop": nested_loop(50, 50, 3),
        "Alternating": alternating_branch(2500),
        "Periodic T-T-N": periodic_pattern(2500, [True, True, False]),
        "Random 50/50": random_branch(2500, 0.5, 0x5000, 2, 1),
        "Random 85/15": random_branch(2500, 0.85, 0x5500, 2, 2),
        "Matrix Multiply": matrix_multiply_pattern(14),
        "Mixed Workload": mixed_workload(5000, 7),
    }

### Pipeline Simulator and Benchmark Runner

These functions run the simulation using a given predictor and trace, calculating performance metrics, and then execute a full benchmark across all defined predictors and trace patterns.

In [5]:
def run_pipeline(predictor, trace, penalty=4):
    predictor.reset()
    cycles = 0
    mispredictions = 0
    branches = 0

    for e in trace:
        cycles += 1
        if e['isBranch']:
            branches += 1
            predicted = predictor.evaluate(e['pc'], e['taken'], e.get('offset'))
            if predicted != e['taken']:
                mispredictions += 1
                cycles += penalty

    instr = len(trace)
    return {
        'name': predictor.name,
        'category': predictor.category,
        'instructions': instr,
        'branches': branches,
        'mispredictions': mispredictions,
        'accuracy': predictor.accuracy(),
        'cycles': cycles,
        'ipc': instr / cycles if cycles else 0,
        'cpi': cycles / instr if instr else 0,
        'mispredRate': mispredictions / branches if branches else 0,
    }

def run_full_benchmark(penalty=4):
    traces = standard_benchmark_suite()
    rows = []
    for trace_name, trace in traces.items():
        for predictor in build_predictor_suite():
            rows.append({'trace': trace_name, **run_pipeline(predictor, trace, penalty)})
    return rows

### Standalone Test / Example Usage

This block demonstrates how to run the full benchmark and print a summary of predictor accuracies, similar to the original JavaScript's standalone test.

In [6]:
if __name__ == "__main__":
    print("Running full benchmark...")
    rows = run_full_benchmark(4)

    by_predictor = {}
    for r in rows:
        by_predictor.setdefault(r['name'], []).append(r['accuracy'])

    summary = []
    for name, accs in by_predictor.items():
        avg_acc = (sum(accs) / len(accs)) * 100
        summary.append({'name': name, 'avgAcc': f"{avg_acc:.2f}"})

    summary.sort(key=lambda x: float(x['avgAcc']), reverse=True)

    print("\n=== Average accuracy by predictor ===")
    for s in summary:
        print(f"{s['name']:<28} {s['avgAcc']}%")

    print("\n=== Alternating trace sanity check (should break 2-bit counter) ===")
    alt_trace = alternating_branch(2500)
    for p in build_predictor_suite():
        r = run_pipeline(p, alt_trace, 4)
        print(f"{p.name:<28} {r['accuracy']*100:.1f}%")

Running full benchmark...


AttributeError: module 'math' has no attribute 'imul'

In [1]:
// ============================================================
// Core simulation logic: predictors, trace generators, pipeline
// (tested standalone here with node, then embedded in the app)
// ============================================================

// ---------------------------------------------------------------
// Seeded RNG (mulberry32) so traces are reproducible
// ---------------------------------------------------------------
function mulberry32(seed) {
  return function () {
    seed |= 0; seed = (seed + 0x6D2B79F5) | 0;
    let t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
    t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
    return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
  };
}

// ---------------------------------------------------------------
// Predictors
// ---------------------------------------------------------------
class Predictor {
  constructor(name) { this.name = name; this.correct = 0; this.total = 0; }
  predict(pc, offset) { throw new Error("not implemented"); }
  update(pc, actual, predicted) { throw new Error("not implemented"); }
  evaluate(pc, actual, offset) {
    const predicted = this.predict(pc, offset || 0);
    this.total++;
    if (predicted === actual) this.correct++;
    this.update(pc, actual, predicted);
    return predicted;
  }
  accuracy() { return this.total ? this.correct / this.total : 0; }
  reset() { this.correct = 0; this.total = 0; }
}

class AlwaysTaken extends Predictor {
  constructor() { super("Always Taken"); this.category = "static"; }
  predict() { return true; }
  update() {}
}
class AlwaysNotTaken extends Predictor {
  constructor() { super("Always Not Taken"); this.category = "static"; }
  predict() { return false; }
  update() {}
}
class BTFNT extends Predictor {
  constructor() { super("BTFNT (Static)"); this.category = "static"; }
  predict(pc, offset) { return offset < 0; }
  update() {}
}
class OneBit extends Predictor {
  constructor(bits = 10) {
    super("1-Bit Predictor"); this.category = "counter";
    this.size = 1 << bits; this.table = new Uint8Array(this.size);
  }
  idx(pc) { return pc % this.size; }
  predict(pc) { return this.table[this.idx(pc)] === 1; }
  update(pc, actual) { this.table[this.idx(pc)] = actual ? 1 : 0; }
}
class TwoBit extends Predictor {
  constructor(bits = 10) {
    super("2-Bit Saturating Counter"); this.category = "counter";
    this.size = 1 << bits; this.table = new Uint8Array(this.size).fill(1);
  }
  idx(pc) { return pc % this.size; }
  predict(pc) { return this.table[this.idx(pc)] >= 2; }
  update(pc, actual) {
    const i = this.idx(pc);
    this.table[i] = actual ? Math.min(3, this.table[i] + 1) : Math.max(0, this.table[i] - 1);
  }
}
class TwoLevelAdaptive extends Predictor {
  constructor(pcBits = 10, histBits = 8) {
    super("Two-Level Adaptive"); this.category = "history";
    this.pcSize = 1 << pcBits; this.histBits = histBits;
    this.histMask = (1 << histBits) - 1;
    this.localHist = new Uint16Array(this.pcSize);
    this.pht = new Uint8Array(1 << histBits).fill(1);
  }
  pidx(pc) { return pc % this.pcSize; }
  predict(pc) { return this.pht[this.localHist[this.pidx(pc)]] >= 2; }
  update(pc, actual) {
    const p = this.pidx(pc), h = this.localHist[p];
    this.pht[h] = actual ? Math.min(3, this.pht[h] + 1) : Math.max(0, this.pht[h] - 1);
    this.localHist[p] = ((h << 1) | (actual ? 1 : 0)) & this.histMask;
  }
}
class GShare extends Predictor {
  constructor(tableBits = 12, histBits = 12) {
    super("GShare"); this.category = "history";
    this.ghr = 0; this.ghrMask = (1 << histBits) - 1;
    this.pht = new Uint8Array(1 << tableBits).fill(1);
    this.idxMask = (1 << tableBits) - 1;
  }
  idx(pc) { return (pc ^ this.ghr) & this.idxMask; }
  predict(pc) { return this.pht[this.idx(pc)] >= 2; }
  update(pc, actual) {
    const i = this.idx(pc);
    this.pht[i] = actual ? Math.min(3, this.pht[i] + 1) : Math.max(0, this.pht[i] - 1);
    this.ghr = ((this.ghr << 1) | (actual ? 1 : 0)) & this.ghrMask;
  }
}
class Tournament extends Predictor {
  constructor(tableBits = 12, histBits = 12) {
    super("Tournament (Hybrid)"); this.category = "history";
    this.local = new TwoLevelAdaptive(tableBits, 10);
    this.gshare = new GShare(tableBits, histBits);
    this.meta = new Uint8Array(1 << histBits).fill(1);
    this.metaMask = (1 << histBits) - 1;
  }
  midx() { return this.gshare.ghr & this.metaMask; }
  predict(pc) {
    this._lLocal = this.local.predict(pc);
    this._lGshare = this.gshare.predict(pc);
    return this.meta[this.midx()] >= 2 ? this._lGshare : this._lLocal;
  }
  update(pc, actual) {
    const m = this.midx();
    const lc = this._lLocal === actual, gc = this._lGshare === actual;
    if (gc && !lc) this.meta[m] = Math.min(3, this.meta[m] + 1);
    else if (lc && !gc) this.meta[m] = Math.max(0, this.meta[m] - 1);
    this.local.update(pc, actual, this._lLocal);
    this.gshare.update(pc, actual, this._lGshare);
  }
}
class PerceptronPredictor extends Predictor {
  constructor(tableSize = 512, histLen = 24) {
    super("Perceptron (AI)"); this.category = "ai";
    this.tableSize = tableSize; this.histLen = histLen;
    this.weights = Array.from({ length: tableSize }, () => new Int16Array(histLen + 1));
    this.ghr = new Int8Array(histLen);
    this.theta = Math.floor(1.93 * histLen + 14);
  }
  idx(pc) { return pc % this.tableSize; }
  dot(w) { let y = w[0]; for (let i = 0; i < this.histLen; i++) y += w[i + 1] * this.ghr[i]; return y; }
  predict(pc) { const w = this.weights[this.idx(pc)]; this._y = this.dot(w); return this._y >= 0; }
  update(pc, actual, predicted) {
    const w = this.weights[this.idx(pc)];
    const t = actual ? 1 : -1;
    if (predicted !== actual || Math.abs(this._y) <= this.theta) {
      w[0] += t;
      for (let i = 0; i < this.histLen; i++) w[i + 1] += t * this.ghr[i];
    }
    for (let i = this.histLen - 1; i > 0; i--) this.ghr[i] = this.ghr[i - 1];
    this.ghr[0] = t;
  }
}
class NeuralMLP extends Predictor {
  constructor(histLen = 16, hidden = 8, lr = 0.15, seed = 42) {
    super("Neural MLP (AI)"); this.category = "ai";
    this.h = histLen; this.hidden = hidden; this.lr = lr;
    const rnd = mulberry32(seed);
    this.W1 = Array.from({ length: hidden }, () => Array.from({ length: histLen }, () => (rnd() - 0.5) * 0.6));
    this.b1 = new Float64Array(hidden);
    this.W2 = Array.from({ length: hidden }, () => (rnd() - 0.5) * 0.6);
    this.b2 = 0;
    this.ghr = new Float64Array(histLen);
  }
  sigmoid(x) { if (x < -60) return 0; if (x > 60) return 1; return 1 / (1 + Math.exp(-x)); }
  forward() {
    const a1 = new Float64Array(this.hidden);
    for (let j = 0; j < this.hidden; j++) {
      let s = this.b1[j];
      const row = this.W1[j];
      for (let i = 0; i < this.h; i++) s += row[i] * this.ghr[i];
      a1[j] = Math.tanh(s);
    }
    let z2 = this.b2;
    for (let j = 0; j < this.hidden; j++) z2 += this.W2[j] * a1[j];
    return { a1, y: this.sigmoid(z2) };
  }
  predict() { const { a1, y } = this.forward(); this._a1 = a1; this._y = y; return y >= 0.5; }
  update(pc, actual) {
    const target = actual ? 1 : 0;
    const err = this._y - target;
    for (let j = 0; j < this.hidden; j++) this.W2[j] -= this.lr * err * this._a1[j];
    this.b2 -= this.lr * err;
    for (let j = 0; j < this.hidden; j++) {
      const d = err * this.W2[j] * (1 - this._a1[j] * this._a1[j]);
      for (let i = 0; i < this.h; i++) this.W1[j][i] -= this.lr * d * this.ghr[i];
      this.b1[j] -= this.lr * d;
    }
    const t = actual ? 1 : -1;
    for (let i = this.h - 1; i > 0; i--) this.ghr[i] = this.ghr[i - 1];
    this.ghr[0] = t;
  }
}

function buildPredictorSuite() {
  return [
    new AlwaysTaken(), new AlwaysNotTaken(), new BTFNT(),
    new OneBit(), new TwoBit(),
    new TwoLevelAdaptive(), new GShare(), new Tournament(),
    new PerceptronPredictor(), new NeuralMLP(),
  ];
}

// ---------------------------------------------------------------
// Trace generators
// ---------------------------------------------------------------
function tightLoop(iterations, bodyLen = 4, basePc = 0x1000) {
  const trace = []; let pc = basePc; const branchPc = basePc + bodyLen * 4;
  for (let i = 0; i < iterations; i++) {
    for (let k = 0; k < bodyLen; k++) { trace.push({ pc, isBranch: false }); pc += 4; }
    trace.push({ pc: branchPc, isBranch: true, taken: i < iterations - 1, offset: -bodyLen * 4 });
    pc = branchPc + 4;
  }
  return trace;
}
function nestedLoop(outer, inner, bodyLen = 3, basePc = 0x2000) {
  const trace = []; let pc = basePc;
  const outerPc = basePc + 0x100, innerPc = basePc + 0x040;
  for (let i = 0; i < outer; i++) {
    for (let j = 0; j < inner; j++) {
      for (let k = 0; k < bodyLen; k++) { trace.push({ pc, isBranch: false }); pc += 4; }
      trace.push({ pc: innerPc, isBranch: true, taken: j < inner - 1, offset: -bodyLen * 4 });
    }
    trace.push({ pc: outerPc, isBranch: true, taken: i < outer - 1, offset: -(inner * (bodyLen + 1)) * 4 });
  }
  return trace;
}
function alternatingBranch(length, basePc = 0x3000, bodyLen = 2) {
  const trace = []; let pc = basePc; const branchPc = basePc + bodyLen * 4;
  for (let i = 0; i < length; i++) {
    for (let k = 0; k < bodyLen; k++) { trace.push({ pc, isBranch: false }); pc += 4; }
    trace.push({ pc: branchPc, isBranch: true, taken: i % 2 === 0, offset: -bodyLen * 4 });
    pc = branchPc + 4;
  }
  return trace;
}
function periodicPattern(length, pattern, basePc = 0x4000, bodyLen = 2) {
  const trace = []; let pc = basePc; const branchPc = basePc + bodyLen * 4;
  for (let i = 0; i < length; i++) {
    for (let k = 0; k < bodyLen; k++) { trace.push({ pc, isBranch: false }); pc += 4; }
    trace.push({ pc: branchPc, isBranch: true, taken: pattern[i % pattern.length], offset: -bodyLen * 4 });
    pc = branchPc + 4;
  }
  return trace;
}
function randomBranch(length, pTaken = 0.5, basePc = 0x5000, bodyLen = 2, seed = 1) {
  const rnd = mulberry32(seed);
  const trace = []; let pc = basePc; const branchPc = basePc + bodyLen * 4;
  for (let i = 0; i < length; i++) {
    for (let k = 0; k < bodyLen; k++) { trace.push({ pc, isBranch: false }); pc += 4; }
    trace.push({ pc: branchPc, isBranch: true, taken: rnd() < pTaken, offset: bodyLen * 4 });
    pc = branchPc + 4;
  }
  return trace;
}
function matrixMultiplyPattern(n, basePc = 0x6000) {
  const trace = []; let pc = basePc;
  const pcI = basePc + 0x300, pcJ = basePc + 0x200, pcK = basePc + 0x100;
  for (let i = 0; i < n; i++) {
    for (let j = 0; j < n; j++) {
      for (let k = 0; k < n; k++) {
        trace.push({ pc, isBranch: false }); pc += 4;
        trace.push({ pc, isBranch: false }); pc += 4;
        trace.push({ pc: pcK, isBranch: true, taken: k < n - 1, offset: -8 });
      }
      trace.push({ pc: pcJ, isBranch: true, taken: j < n - 1, offset: -8 * n });
    }
    trace.push({ pc: pcI, isBranch: true, taken: i < n - 1, offset: -8 * n * n });
  }
  return trace;
}
function mixedWorkload(totalLength = 4000, seed = 7) {
  const chunk = Math.floor(totalLength / 6);
  let trace = [];
  trace = trace.concat(tightLoop(Math.floor(chunk / 4), 4, 0x1000));
  const s = Math.floor(Math.sqrt(chunk / 6)) + 2;
  trace = trace.concat(nestedLoop(s, s, 3, 0x2000));
  trace = trace.concat(alternatingBranch(chunk, 0x3000));
  trace = trace.concat(periodicPattern(chunk, [true, true, false], 0x4000));
  trace = trace.concat(randomBranch(chunk, 0.5, 0x5000, 2, seed));
  trace = trace.concat(randomBranch(chunk, 0.85, 0x5500, 2, seed + 1));
  return trace;
}
function standardBenchmarkSuite() {
  return {
    "Tight Loop": tightLoop(1500, 4),
    "Nested Loop": nestedLoop(50, 50, 3),
    "Alternating": alternatingBranch(2500),
    "Periodic T-T-N": periodicPattern(2500, [true, true, false]),
    "Random 50/50": randomBranch(2500, 0.5, 0x5000, 2, 1),
    "Random 85/15": randomBranch(2500, 0.85, 0x5500, 2, 2),
    "Matrix Multiply": matrixMultiplyPattern(14),
    "Mixed Workload": mixedWorkload(5000, 7),
  };
}

// ---------------------------------------------------------------
// Pipeline simulator
// ---------------------------------------------------------------
function runPipeline(predictor, trace, penalty = 4) {
  predictor.reset();
  let cycles = 0, mispredictions = 0, branches = 0;
  for (const e of trace) {
    cycles++;
    if (e.isBranch) {
      branches++;
      const predicted = predictor.evaluate(e.pc, e.taken, e.offset || 0);
      if (predicted !== e.taken) { mispredictions++; cycles += penalty; }
    }
  }
  const instr = trace.length;
  return {
    name: predictor.name, category: predictor.category,
    instructions: instr, branches, mispredictions,
    accuracy: predictor.accuracy(), cycles,
    ipc: instr / cycles, cpi: cycles / instr,
    mispredRate: branches ? mispredictions / branches : 0,
  };
}

function runFullBenchmark(penalty = 4) {
  const traces = standardBenchmarkSuite();
  const rows = [];
  for (const [traceName, trace] of Object.entries(traces)) {
    for (const predictor of buildPredictorSuite()) {
      rows.push({ trace: traceName, ...runPipeline(predictor, trace, penalty) });
    }
  }
  return rows;
}

// ---------------------------------------------------------------
// Standalone test
// ---------------------------------------------------------------
if (typeof require !== "undefined" && require.main === module) {
  const rows = runFullBenchmark(4);
  const byPredictor = {};
  for (const r of rows) {
    byPredictor[r.name] = byPredictor[r.name] || [];
    byPredictor[r.name].push(r.accuracy);
  }
  const summary = Object.entries(byPredictor).map(([name, accs]) => ({
    name, avgAcc: (accs.reduce((a, b) => a + b, 0) / accs.length * 100).toFixed(2),
  })).sort((a, b) => b.avgAcc - a.avgAcc);
  console.log("=== Average accuracy by predictor ===");
  for (const s of summary) console.log(s.name.padEnd(28), s.avgAcc + "%");

  console.log("\n=== Alternating trace sanity check (should break 2-bit counter) ===");
  const altTrace = alternatingBranch(2500);
  for (const p of buildPredictorSuite()) {
    const r = runPipeline(p, altTrace, 4);
    console.log(p.name.padEnd(28), (r.accuracy * 100).toFixed(1) + "%");
  }
}

module.exports = {
  buildPredictorSuite, standardBenchmarkSuite, runPipeline, runFullBenchmark,
  tightLoop, nestedLoop, alternatingBranch, periodicPattern, randomBranch,
  matrixMultiplyPattern, mixedWorkload,
};

SyntaxError: invalid syntax (3691643237.py, line 1)